# Extension modelling: Sub-ICB emergency admissions (2022-2024)

This notebook builds the extension analysis reported in Chapter 4 of the dissertation.
Data sources are HES Monthly Activity Report (emergency admissions target),
Fingertips QOF prevalence indicators, and NHS Digital GP appointments. All work
is at Sub-ICB Location level from April 2022 to December 2024.

Sections:
1. Load data and check for duplicates
2. Check Fingertips coverage
3. Correlations with the target
4. Define feature sets and analysis sample
5. Hyperparameter tuning
6. Model comparison across feature sets
7. Feature importance
8. MGSR stacking
9. Flat model vs MGSR
10. GroupKFold validation
11. Diagnostic checks
12. 2024 forecasting with lagged features
13. Time-series baselines
14. Summary and figures

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (RepeatedKFold, GroupKFold, KFold,
                                     cross_val_predict, RandomizedSearchCV)
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from scipy.stats import wilcoxon, ttest_ind, randint, uniform
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

DATA = Path('../new_data')
FIG = Path('../figures')

SEED = 0
target = 'emergency_admissions_rate'
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep',
               'Oct','Nov','Dec']

cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

## 1. Load data and check for duplicates

The HES CSV has some duplicated `(sub_icb, year, month)` rows from overlapping
annual files. Drop them before modelling.

In [4]:
raw = pd.read_csv(DATA / 'master_hes_2022_2025.csv')
raw = raw.rename(columns={'ccg': 'sub_icb'})

# check duplicates
n_dup = raw.duplicated(['sub_icb', 'year', 'month']).sum()
print(f'Duplicate rows: {n_dup}')

# drop them
data = raw.drop_duplicates(['sub_icb', 'year', 'month']).reset_index(drop=True)

# build date and seasonal features
data['month_num'] = data['month'].map({m: i+1 for i, m in enumerate(month_order)})
data['month_sin'] = np.sin(2 * np.pi * data['month_num'] / 12)
data['month_cos'] = np.cos(2 * np.pi * data['month_num'] / 12)
data['date'] = pd.to_datetime(dict(year=data.year.astype(int),
                                   month=data.month_num.astype(int), day=1))
data = data.dropna(subset=[target]).sort_values(['sub_icb', 'date']).reset_index(drop=True)

# QOF health columns
fx_cols = [c for c in data.columns if c.startswith('fx_') and 'imd' not in c.lower()]

print(f'After dedup: {len(data)} rows, {data.sub_icb.nunique()} Sub-ICBs, '
      f'{data.icb_name.nunique()} ICBs')
print(f'Date range: {data.date.min().date()} to {data.date.max().date()}')

Duplicate rows: 97
After dedup: 3201 rows, 97 Sub-ICBs, 35 ICBs
Date range: 2022-04-01 to 2024-12-01


## 2. Fingertips coverage

The Fingertips QOF merge only succeeds for around 77% of rows because some
ICBs can't be mapped through the ONS-to-NHS code bridge. Check whether the
dropped Sub-ICBs differ systematically from the kept ones.

In [6]:
# feature groups
HEALTH = [c for c in fx_cols if data[c].notna().mean() > 0.5]
DEMOG = ['pct_65plus', 'population']
GP = ['gp_appt_available_rate']
ACTIVITY = [c for c in ['elective_total_rate', 'outpatient_first_total_rate',
                        'outpatient_dna_rate', 'gp_referrals_rate']
            if c in data.columns]
SEASON = ['month_sin', 'month_cos']

# rows with all features present
all_feats = GP + ACTIVITY + HEALTH + DEMOG
complete = data[all_feats + [target]].notna().all(axis=1)

kept_ids = set(data.loc[complete, 'sub_icb'])
dropped_ids = set(data.sub_icb) - kept_ids

print(f'Sub-ICBs kept: {len(kept_ids)}, dropped: {len(dropped_ids)}')
print(f'Rows kept: {complete.sum()} of {len(data)} '
      f'({complete.mean() * 100:.1f}%)')

# compare kept vs dropped Sub-ICBs
area_stats = data.groupby('sub_icb').agg(
    target_mean=(target, 'mean'),
    population=('population', 'mean'),
    pct_65plus=('pct_65plus', 'mean')
).reset_index()
area_stats['kept'] = area_stats.sub_icb.isin(kept_ids)

print('\nMean values by group:')
print(area_stats.groupby('kept')[['target_mean', 'population', 'pct_65plus']].mean().round(2))

print('\nt-tests (kept vs dropped):')
for col in ['target_mean', 'population', 'pct_65plus']:
    a = area_stats.loc[area_stats.kept, col].dropna()
    b = area_stats.loc[~area_stats.kept, col].dropna()
    t, p = ttest_ind(a, b, equal_var=False)
    print(f'  {col}: t = {t:.2f}, p = {p:.3f}')

Sub-ICBs kept: 75, dropped: 22
Rows kept: 2325 of 3201 (72.6%)

Mean values by group:
       target_mean  population  pct_65plus
kept                                      
False       112.07       54.82       19.36
True        130.40       46.67       19.99

t-tests (kept vs dropped):
  target_mean: t = 4.77, p = 0.000
  population: t = -0.94, p = 0.352
  pct_65plus: t = 0.67, p = 0.508


## 3. Correlations with the target

Because elective admissions and the target both come from HES and are both
scaled by the same population variable, some of the raw correlation could
be an artefact of the shared denominator. Looking at three versions:

- correlation of the rates
- partial correlation controlling for population
- within-Sub-ICB correlation (both series de-meaned by area)

In [7]:
check_cols = GP + ACTIVITY + HEALTH + DEMOG
if '111_offered_rate' in data.columns:
    check_cols += ['111_offered_rate']

corrs = data[check_cols + [target]].corr()[target].drop(target)
corrs = corrs.sort_values(key=abs, ascending=False)
print('Correlation with emergency admission rate:')
print(corrs.round(3))

# shared-denominator check for activity features
print('\nShared-denominator check (activity features):')
print(f"{'feature':30s}  r(rate)  partial|pop  within-area")
for c in ACTIVITY + GP:
    s = data[[c, target, 'population', 'sub_icb']].dropna()
    r_rate = s[c].corr(s[target])

    # partial correlation controlling for population
    pop = s[['population']].values
    resid_x = s[c].values - LinearRegression().fit(pop, s[c]).predict(pop)
    resid_y = s[target].values - LinearRegression().fit(pop, s[target]).predict(pop)
    r_partial = np.corrcoef(resid_x, resid_y)[0, 1]

    # within-area (both series de-meaned by Sub-ICB)
    x_dm = s[c] - s.groupby('sub_icb')[c].transform('mean')
    y_dm = s[target] - s.groupby('sub_icb')[target].transform('mean')
    r_within = x_dm.corr(y_dm)

    print(f'{c:30s}  {r_rate:>6.3f}  {r_partial:>10.3f}  {r_within:>10.3f}')

Correlation with emergency admission rate:
elective_total_rate                 0.527
fx_copd__qof_prevalence             0.347
fx_hypertension__qof_prevalence     0.320
fx_heart_failure__qof_prevalence    0.309
fx_chd__qof_prevalence              0.291
population                         -0.278
gp_appt_available_rate              0.236
outpatient_dna_rate                 0.222
fx_diabetes__qof_prevalence         0.221
fx_asthma__qof_prevalence           0.203
fx_depression__qof_prevalence       0.191
pct_65plus                          0.130
outpatient_first_total_rate         0.074
gp_referrals_rate                   0.049
111_offered_rate                   -0.005
Name: emergency_admissions_rate, dtype: float64

Shared-denominator check (activity features):
feature                         r(rate)  partial|pop  within-area
elective_total_rate              0.527       0.480       0.497
outpatient_first_total_rate      0.074       0.157       0.353
outpatient_dna_rate              0.222  

## 4. Analysis sample and feature sets

Use one complete-case dataframe throughout, so every feature set is scored
on identical rows.

In [8]:
clean = data[complete].reset_index(drop=True)
print(f'Analysis sample: {len(clean)} rows, '
      f'{clean.sub_icb.nunique()} Sub-ICBs, '
      f'{clean.icb_name.nunique()} ICBs')

feature_sets = {
    'Health only (QOF)':         HEALTH,
    'GP only':                   GP,
    'GP + Health':               GP + HEALTH,
    'Demographics only':         DEMOG,
    'GP + Health + Demog':       GP + HEALTH + DEMOG,
    'GP + Health + Demog + Sea': GP + HEALTH + DEMOG + SEASON,
    'Service Activity':          GP + ACTIVITY + SEASON,
    'All features':              GP + ACTIVITY + HEALTH + DEMOG + SEASON,}

# fix CV splits so every model uses the same folds
splits = list(cv.split(clean))
print(f'{len(splits)} folds')

Analysis sample: 2325 rows, 75 Sub-ICBs, 25 ICBs
15 folds


## 5. Hyperparameter tuning

Tune the tree ensembles once per role using `RandomizedSearchCV`. Tuning
inside each downstream cross-validation loop would be slow and not really
justified at this sample size.

In [10]:
param_gbr = {
    'n_estimators':  randint(100, 500),
    'max_depth':     randint(2, 6),
    'learning_rate': uniform(0.01, 0.19),
    'subsample':     uniform(0.6, 0.4),}
param_rf = {
    'n_estimators':     randint(100, 500),
    'max_depth':        randint(2, 10),
    'min_samples_leaf': randint(1, 10),
    'max_features':     uniform(0.3, 0.7),}
param_xgb = {
    'n_estimators':     randint(100, 500),
    'max_depth':        randint(2, 6),
    'learning_rate':    uniform(0.01, 0.19),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),}

def tune(estimator, params, feats):
    search = RandomizedSearchCV(estimator, params, n_iter=30, cv=cv,
                                scoring='r2', random_state=SEED, n_jobs=-1)
    search.fit(clean[feats], clean[target])
    return search.best_params_, search.best_score_

# tune full-feature models
print('Tuning full-feature models...')
full_feats = feature_sets['All features']
gbr_full_p, gbr_full_s = tune(GradientBoostingRegressor(random_state=SEED), param_gbr, full_feats)
rf_full_p,  rf_full_s  = tune(RandomForestRegressor(random_state=SEED), param_rf, full_feats)
xgb_full_p, xgb_full_s = tune(XGBRegressor(random_state=SEED, verbosity=0), param_xgb, full_feats)
print(f'  GBR: R2 = {gbr_full_s:.4f}')
print(f'  RF:  R2 = {rf_full_s:.4f}')
print(f'  XGB: R2 = {xgb_full_s:.4f}')

# tune Level-0a models (Service Activity block for MGSR)
print('\nTuning Level-0a (Service Activity)...')
l0a_feats = feature_sets['Service Activity']
gbr_l0a_p, gbr_l0a_s = tune(GradientBoostingRegressor(random_state=SEED), param_gbr, l0a_feats)
rf_l0a_p,  rf_l0a_s  = tune(RandomForestRegressor(random_state=SEED), param_rf, l0a_feats)
print(f'  GBR: R2 = {gbr_l0a_s:.4f}')
print(f'  RF:  R2 = {rf_l0a_s:.4f}')

# tune Level-0b models (Health + Demographics)
print('\nTuning Level-0b (Health + Demographics)...')
l0b_feats = HEALTH + DEMOG
gbr_l0b_p, gbr_l0b_s = tune(GradientBoostingRegressor(random_state=SEED), param_gbr, l0b_feats)
rf_l0b_p,  rf_l0b_s  = tune(RandomForestRegressor(random_state=SEED), param_rf, l0b_feats)
print(f'  GBR: R2 = {gbr_l0b_s:.4f}')
print(f'  RF:  R2 = {rf_l0b_s:.4f}')

Tuning full-feature models...
  GBR: R2 = 0.9111
  RF:  R2 = 0.8537
  XGB: R2 = 0.9153

Tuning Level-0a (Service Activity)...
  GBR: R2 = 0.5674
  RF:  R2 = 0.5608

Tuning Level-0b (Health + Demographics)...
  GBR: R2 = 0.8566
  RF:  R2 = 0.8432


## 6. Model comparison across feature sets

Also compute two baselines: the global mean and the group mean (each
Sub-ICB predicted by its own historical average). A model that doesn't
beat the group mean has only learned which area each row is from.

In [11]:
models = {
    'Ridge':      make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso':      make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5)),
    'KNN':        KNeighborsRegressor(n_neighbors=10),
    'SVR':        make_pipeline(StandardScaler(), SVR(kernel='rbf', C=10)),
    'RF':         RandomForestRegressor(random_state=SEED, **rf_full_p),
    'GBR':        GradientBoostingRegressor(random_state=SEED, **gbr_full_p),
    'XGB':        XGBRegressor(random_state=SEED, verbosity=0, **xgb_full_p),}

def score_folds(feats, model):
    scores = []
    for tr, te in splits:
        m = clone(model).fit(clean.iloc[tr][feats], clean.iloc[tr][target])
        pred = m.predict(clean.iloc[te][feats])
        scores.append(r2_score(clean.iloc[te][target], pred))
    return np.array(scores)

# score every model on every feature set
fold_scores = {}
rows = []
for mname, model in models.items():
    row = {'model': mname}
    for fname, feats in feature_sets.items():
        s = score_folds(feats, model)
        fold_scores[(mname, fname)] = s
        row[fname] = s.mean()
    rows.append(row)

results = pd.DataFrame(rows).set_index('model')
print('Cross-validated R2 (5-fold x 3 repeats):')
print(results.round(4))

# baselines
group_mean_scores = []
global_mean_scores = []
for tr, te in splits:
    trd = clean.iloc[tr]
    ted = clean.iloc[te]
    # group mean: predict each Sub-ICB by its own training-set mean
    area_means = trd.groupby('sub_icb')[target].mean()
    pred_group = ted.sub_icb.map(area_means).fillna(trd[target].mean())
    group_mean_scores.append(r2_score(ted[target], pred_group))
    # global mean
    pred_global = np.full(len(ted), trd[target].mean())
    global_mean_scores.append(r2_score(ted[target], pred_global))

group_mean_scores = np.array(group_mean_scores)
global_mean_scores = np.array(global_mean_scores)

print(f'\nBaselines:')
print(f'  Global mean:  R2 = {global_mean_scores.mean():.4f}')
print(f'  Group mean:   R2 = {group_mean_scores.mean():.4f}')

# Wilcoxon tests: does each model beat the group-mean baseline?
print('\nWilcoxon tests against group-mean baseline:')
for mname in ['RF', 'GBR', 'XGB']:
    for fname in ['GP + Health + Demog + Sea', 'Service Activity', 'All features']:
        s = fold_scores[(mname, fname)]
        p = wilcoxon(s, group_mean_scores).pvalue
        print(f'  {mname} on {fname}: R2 = {s.mean():.4f}, '
              f'delta = {s.mean() - group_mean_scores.mean():+.4f}, p = {p:.4f}')

Cross-validated R2 (5-fold x 3 repeats):
            Health only (QOF)  GP only  GP + Health  Demographics only  \
model                                                                    
Ridge                  0.1603   0.0418       0.1961             0.0707   
Lasso                  0.1607   0.0418       0.1965             0.0707   
ElasticNet             0.1600   0.0417       0.1956             0.0706   
KNN                    0.0668  -0.0661      -0.0526             0.8430   
SVR                    0.2915   0.0275       0.3981             0.1660   
RF                     0.3173  -0.0257       0.4321             0.8002   
GBR                    0.3177  -0.3209       0.2775             0.8558   
XGB                    0.3174  -0.0849       0.3640             0.8548   

            GP + Health + Demog  GP + Health + Demog + Sea  Service Activity  \
model                                                                          
Ridge                    0.2220                     0.2216

## 7. Feature importance and leave-one-out

In [14]:
best_feats = feature_sets['All features']
best_scores = score_folds(best_feats, models['GBR'])
print(f"GBR on all features: R2 = {best_scores.mean():.4f}")

# feature importance from RF
rf = RandomForestRegressor(random_state=SEED, **rf_full_p)
rf.fit(clean[best_feats], clean[target])
importance = pd.Series(rf.feature_importances_, index=best_feats).sort_values(ascending=False)
print('\nFeature importance (RF):')
print(importance.round(4))

# leave-one-feature-out
print('\nLeave-one-feature-out (drop in R2 when feature is removed):')
loo_rows = []
for f in best_feats:
    remaining = [x for x in best_feats if x != f]
    s = score_folds(remaining, models['GBR'])
    delta = best_scores.mean() - s.mean()
    p = wilcoxon(best_scores, s).pvalue
    loo_rows.append({'feature': f, 'delta_R2': delta, 'p': p})

loo = pd.DataFrame(loo_rows).sort_values('delta_R2', ascending=False)
print(loo.round(4).to_string(index=False))

GBR on all features: R2 = 0.9111

Feature importance (RF):
elective_total_rate                 0.2080
pct_65plus                          0.1702
outpatient_dna_rate                 0.1244
fx_chd__qof_prevalence              0.0979
population                          0.0844
fx_copd__qof_prevalence             0.0789
fx_asthma__qof_prevalence           0.0372
gp_appt_available_rate              0.0344
fx_diabetes__qof_prevalence         0.0318
outpatient_first_total_rate         0.0315
gp_referrals_rate                   0.0312
fx_heart_failure__qof_prevalence    0.0280
fx_depression__qof_prevalence       0.0210
fx_hypertension__qof_prevalence     0.0148
month_sin                           0.0033
month_cos                           0.0031
dtype: float64

Leave-one-feature-out (drop in R2 when feature is removed):
                         feature  delta_R2      p
                      pct_65plus    0.0288 0.0001
             elective_total_rate    0.0097 0.2078
                       mont

## 8. MGSR stacking

Run the two-level stack with two definitions of the Level-0a block:
- **GP only** (closest to the paper's capacity block)
- **Service Activity** (GP + secondary-care activity measures)

Meta-learner is trained on out-of-fold Level-0 predictions to avoid leakage.

In [26]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

def run_mgsr(a_feats, b_feats, a_params, b_params, learner):
    sa, sb, sm, coefs = [], [], [], []
    for tr, te in splits:
        trd = clean.iloc[tr]
        ted = clean.iloc[te]

        # out-of-fold predictions for training the meta-learner
        a_tr = cross_val_predict(learner(random_state=SEED, **a_params),
                                 trd[a_feats], trd[target], cv=inner_cv)
        b_tr = cross_val_predict(learner(random_state=SEED, **b_params),
                                 trd[b_feats], trd[target], cv=inner_cv)

        # test-fold predictions from models fitted on the full training fold
        a_model = learner(random_state=SEED, **a_params).fit(trd[a_feats], trd[target])
        b_model = learner(random_state=SEED, **b_params).fit(trd[b_feats], trd[target])
        a_te = a_model.predict(ted[a_feats])
        b_te = b_model.predict(ted[b_feats])

        sa.append(r2_score(ted[target], a_te))
        sb.append(r2_score(ted[target], b_te))

        meta = LinearRegression().fit(np.column_stack([a_tr, b_tr]), trd[target])
        stack_pred = meta.predict(np.column_stack([a_te, b_te]))
        sm.append(r2_score(ted[target], stack_pred))
        coefs.append(meta.coef_)

    return np.array(sa), np.array(sb), np.array(sm), np.array(coefs).mean(axis=0)


l0a_defs = {
    'GP only (as in paper)':       GP + SEASON,
    'Service Activity (extended)': GP + ACTIVITY + SEASON,
}
l0b_feats = HEALTH + DEMOG

mgsr_scores = {}
for cfg_name, a_feats in l0a_defs.items():
    print(f'\nLevel-0a: {cfg_name}')
    for lname, learner, a_p, b_p in [
        ('RF + RF',   RandomForestRegressor,     rf_l0a_p,  rf_l0b_p),
        ('GBR + GBR', GradientBoostingRegressor, gbr_l0a_p, gbr_l0b_p),
    ]:
        sa, sb, sm, coefs = run_mgsr(a_feats, l0b_feats, a_p, b_p, learner)
        mgsr_scores[(cfg_name, lname)] = sm
        print(f'  {lname}: L0a = {sa.mean():.4f}, L0b = {sb.mean():.4f}, '
              f'stack = {sm.mean():.4f}')
        print(f'    OLS weights: L0a = {coefs[0]:.3f}, L0b = {coefs[1]:.3f}')


Level-0a: GP only (as in paper)
  RF + RF: L0a = -0.0263, L0b = 0.8432, stack = 0.8500
    OLS weights: L0a = 0.178, L0b = 1.054
  GBR + GBR: L0a = -0.0447, L0b = 0.8566, stack = 0.8593
    OLS weights: L0a = 0.131, L0b = 0.979

Level-0a: Service Activity (extended)
  RF + RF: L0a = 0.5608, L0b = 0.8432, stack = 0.8630
    OLS weights: L0a = 0.294, L0b = 0.900
  GBR + GBR: L0a = 0.5674, L0b = 0.8566, stack = 0.8752
    OLS weights: L0a = 0.267, L0b = 0.837


## 9. Flat model vs MGSR

Test whether the two-level stack actually helps against a flat single
model trained on the union of both feature blocks.

In [27]:
flat_feats = sorted(set(GP + ACTIVITY + SEASON + l0b_feats))
print(f'Flat model: {len(flat_feats)} features\n')

for lname, model in [('RF', models['RF']), ('GBR', models['GBR'])]:
    stack = mgsr_scores[('Service Activity (extended)', f'{lname} + {lname}')]
    flat = score_folds(flat_feats, model)
    p = wilcoxon(flat, stack).pvalue
    print(f'{lname}:')
    print(f'  MGSR stack: R2 = {stack.mean():.4f}')
    print(f'  Flat model: R2 = {flat.mean():.4f}')
    print(f'  Difference: {flat.mean() - stack.mean():+.4f}, p = {p:.4f}')
    print()

Flat model: 16 features

RF:
  MGSR stack: R2 = 0.8630
  Flat model: R2 = 0.8521
  Difference: -0.0108, p = 0.0730

GBR:
  MGSR stack: R2 = 0.8752
  Flat model: R2 = 0.9123
  Difference: +0.0370, p = 0.0001



## 10. GroupKFold validation

Holding out whole geographies to check whether the model generalises to
areas it hasn't seen. GroupKFold by Sub-ICB holds out individual
Sub-ICBs; by ICB is stricter since the Fingertips health features are
identical for every Sub-ICB inside an ICB.

In [ ]:

for group_col, label in [('sub_icb', 'by Sub-ICB'), ('icb_name', 'by ICB')]:
    n_groups = clean[group_col].nunique()
    grouped_splits = list(GroupKFold(n_splits=5).split(
        clean, clean[target], clean[group_col]))
    print(f'\nGroupKFold {label} ({n_groups} groups):')

    for fname in ['Demographics only', 'GP + Health + Demog + Sea',
                  'Service Activity', 'All features']:
        random_s = fold_scores[('GBR', fname)]
        grouped_s = []
        for tr, te in grouped_splits:
            m = clone(models['GBR']).fit(
                clean.iloc[tr][feature_sets[fname]], clean.iloc[tr][target])
            pred = m.predict(clean.iloc[te][feature_sets[fname]])
            grouped_s.append(r2_score(clean.iloc[te][target], pred))
        grouped_s = np.array(grouped_s)
        if fname == 'All features':
            if group_col == 'sub_icb':
                r2_group_subicb = grouped_s.mean()
        else:
            r2_group_icb = grouped_s.mean()
        print(f'  {fname}: random = {random_s.mean():+.4f}, '
              f'grouped = {grouped_s.mean():+.4f}')

    # baseline under the same grouped splits
    baseline = []
    for tr, te in grouped_splits:
        trd = clean.iloc[tr]
        ted = clean.iloc[te]
        baseline.append(r2_score(ted[target], np.full(len(ted), trd[target].mean())))
    print(f'  Global-mean baseline (grouped): {np.mean(baseline):+.4f}')


GroupKFold by Sub-ICB (75 groups):
  Demographics only: random = +0.8558, grouped = -0.7341
  GP + Health + Demog + Sea: random = +0.8919, grouped = -0.2988
  Service Activity: random = +0.5587, grouped = +0.0323
  All features: random = +0.9111, grouped = +0.0781
  Global-mean baseline (grouped): -0.1162

GroupKFold by ICB (25 groups):
  Demographics only: random = +0.8558, grouped = -0.6735
  GP + Health + Demog + Sea: random = +0.8919, grouped = -0.7675
  Service Activity: random = +0.5587, grouped = -0.0351
  All features: random = +0.9111, grouped = -0.3851
  Global-mean baseline (grouped): -0.2114


In [19]:
grouped_results  = {}   # {(group_col, fname): grouped mean R^2}
baseline_results = {}   # {group_col: grouped global-mean baseline R^2}

for group_col, label in [('sub_icb', 'by Sub-ICB'), ('icb_name', 'by ICB')]:
    n_groups = clean[group_col].nunique()
    grouped_splits = list(GroupKFold(n_splits=5).split(
        clean, clean[target], clean[group_col]))
    print(f'\nGroupKFold {label} ({n_groups} groups):')

    for fname in ['Demographics only', 'GP + Health + Demog + Sea',
                  'Service Activity', 'All features']:
        random_s = fold_scores[('GBR', fname)]
        grouped_s = []
        for tr, te in grouped_splits:
            m = clone(models['GBR']).fit(
                clean.iloc[tr][feature_sets[fname]], clean.iloc[tr][target])
            pred = m.predict(clean.iloc[te][feature_sets[fname]])
            grouped_s.append(r2_score(clean.iloc[te][target], pred))
        grouped_s = np.array(grouped_s)
        grouped_results[(group_col, fname)] = grouped_s.mean()
        print(f'  {fname}: random = {random_s.mean():+.4f}, '
              f'grouped = {grouped_s.mean():+.4f}')

    # global-mean baseline under the SAME grouped splits
    baseline = []
    for tr, te in grouped_splits:
        trd, ted = clean.iloc[tr], clean.iloc[te]
        baseline.append(r2_score(ted[target], np.full(len(ted), trd[target].mean())))
    baseline_results[group_col] = np.mean(baseline)
    print(f'  Global-mean baseline (grouped): {np.mean(baseline):+.4f}')

# Pull the All-features values the figure and Table 4.7 need
r2_group_subicb = grouped_results[('sub_icb',  'All features')]
r2_group_icb    = grouped_results[('icb_name', 'All features')]
print(f'\nAll-features grouped -> Sub-ICB {r2_group_subicb:+.4f}, '
      f'ICB {r2_group_icb:+.4f}')
# EXPECT: Sub-ICB +0.0781, ICB -0.3851


GroupKFold by Sub-ICB (75 groups):
  Demographics only: random = +0.8558, grouped = -0.7341
  GP + Health + Demog + Sea: random = +0.8919, grouped = -0.2988
  Service Activity: random = +0.5587, grouped = +0.0323
  All features: random = +0.9111, grouped = +0.0781
  Global-mean baseline (grouped): -0.1162

GroupKFold by ICB (25 groups):
  Demographics only: random = +0.8558, grouped = -0.6735
  GP + Health + Demog + Sea: random = +0.8919, grouped = -0.7675
  Service Activity: random = +0.5587, grouped = -0.0351
  All features: random = +0.9111, grouped = -0.3851
  Global-mean baseline (grouped): -0.2114

All-features grouped -> Sub-ICB +0.0781, ICB -0.3851


## 11. Diagnostic checks

- Permutation test: if the fitted relationship is real, R2 should collapse
  to near zero when the target is shuffled.
- Temporal holdout: train on 2022-2023, predict 2024.
- Static features check: do population and age structure act as Sub-ICB
  identifiers? If so, a lot of the reported R2 is memorisation.
- Demographics-only vs full comparison, on the same rows and folds.

In [29]:
# 1. permutation test
shuffled = clean.copy()
shuffled['y_shuffled'] = clean[target].sample(frac=1, random_state=1).values
perm_scores = []
for tr, te in splits:
    m = clone(models['GBR']).fit(shuffled.iloc[tr][best_feats], shuffled.iloc[tr]['y_shuffled'])
    pred = m.predict(shuffled.iloc[te][best_feats])
    perm_scores.append(r2_score(shuffled.iloc[te]['y_shuffled'], pred))
print(f'1. Permutation test: R2 = {np.mean(perm_scores):+.4f} '
      f'(real R2 = {best_scores.mean():.4f})')

# 2. temporal holdout
train_t = clean[clean.year <= 2023]
test_t = clean[clean.year == 2024]
m = clone(models['GBR']).fit(train_t[best_feats], train_t[target])
r2_temporal = r2_score(test_t[target], m.predict(test_t[best_feats]))
print(f'2. Temporal holdout (train <=2023, test 2024): R2 = {r2_temporal:+.4f}')

# 3. A feature is "static" if it has at most one value per Sub-ICB per year
n_years = clean.year.nunique()
values_per_area = clean.groupby('sub_icb')[best_feats].nunique().max()
static_feats = [f for f in best_feats if values_per_area[f] <= n_years]
signatures = clean.groupby(static_feats, dropna=False)['sub_icb'].nunique()
pct_unique = (signatures == 1).mean() * 100
print(f'3. Static features: {static_feats}')
print(f'   {pct_unique:.0f}% of static-feature combinations map to one Sub-ICB')

# ceiling: R2 of the perfect Sub-ICB-mean predictor
ceiling = r2_score(clean[target], clean.groupby('sub_icb')[target].transform('mean'))
print(f'   Sub-ICB-mean ceiling: R2 = {ceiling:.4f}')

# 4. demographics-only vs full
s_demo = score_folds(DEMOG, models['GBR'])
s_no_demo = score_folds([f for f in best_feats if f not in DEMOG], models['GBR'])
print(f'4. Demographics only:            R2 = {s_demo.mean():+.4f}')
print(f'   All features:                 R2 = {best_scores.mean():+.4f}')
print(f'   All features minus demog:     R2 = {s_no_demo.mean():+.4f}')
print(f'   p (full vs demo-only):        {wilcoxon(best_scores, s_demo).pvalue:.4f}')
print(f'   p (full vs no-demog):         {wilcoxon(best_scores, s_no_demo).pvalue:.4f}')

1. Permutation test: R2 = -0.2213 (real R2 = 0.9111)
2. Temporal holdout (train <=2023, test 2024): R2 = +0.5861
3. Static features: ['fx_asthma__qof_prevalence', 'fx_chd__qof_prevalence', 'fx_copd__qof_prevalence', 'fx_depression__qof_prevalence', 'fx_diabetes__qof_prevalence', 'fx_heart_failure__qof_prevalence', 'fx_hypertension__qof_prevalence', 'pct_65plus', 'population']
   100% of static-feature combinations map to one Sub-ICB
   Sub-ICB-mean ceiling: R2 = 0.8245
4. Demographics only:            R2 = +0.8558
   All features:                 R2 = +0.9111
   All features minus demog:     R2 = +0.7932
   p (full vs demo-only):        0.0001
   p (full vs no-demog):         0.0001


## 12. Forecasting 2024 with lagged features

Activity features are lagged by one month (can't use elective volumes from
the month being predicted). Health and demographic features are annual
and available in advance, so used unlagged.

In [30]:
# build lagged features on the full panel
panel = data.sort_values(['sub_icb', 'date']).copy()

lag1 = []
for c in GP + ACTIVITY:
    panel[c + '_lag1'] = panel.groupby('sub_icb')[c].shift(1)
    lag1.append(c + '_lag1')
panel['target_lag1'] = panel.groupby('sub_icb')[target].shift(1)
panel['target_lag12'] = panel.groupby('sub_icb')[target].shift(12)

STATIC = HEALTH + DEMOG + SEASON

specs = {
    'static only':                       STATIC,
    'static + activity lag-1':           STATIC + lag1,
    'static + activity lag-1 + y lag-1': STATIC + lag1 + ['target_lag1'],
}

# fixed test rows: 2024 with all lagged features available
needed = STATIC + lag1 + ['target_lag1', 'target_lag12', target]
test = panel[panel.year == 2024].dropna(subset=needed)
pool = panel[panel.year <= 2023]
print(f'Test rows: {len(test)} ({test.sub_icb.nunique()} Sub-ICBs)')

Test rows: 900 (75 Sub-ICBs)


In [31]:
fc_rows = []

# fit each specification with RF and GBR
for sname, feats in specs.items():
    train = pool.dropna(subset=feats + [target])
    for mname in ['RF', 'GBR']:
        m = clone(models[mname]).fit(train[feats], train[target])
        pred = m.predict(test[feats])
        fc_rows.append({
            'model': f'{mname} ({sname})',
            'R2': r2_score(test[target], pred),
            'MAPE': mean_absolute_percentage_error(test[target], pred) * 100,
        })

# MGSR with lagged Level-0a
a_feats = lag1 + SEASON
b_feats = HEALTH + DEMOG
train = pool.dropna(subset=a_feats + b_feats + [target])

a_oof = cross_val_predict(GradientBoostingRegressor(random_state=SEED, **gbr_l0a_p),
                          train[a_feats], train[target], cv=inner_cv)
b_oof = cross_val_predict(GradientBoostingRegressor(random_state=SEED, **gbr_l0b_p),
                          train[b_feats], train[target], cv=inner_cv)
meta = LinearRegression().fit(np.column_stack([a_oof, b_oof]), train[target])

a_test = GradientBoostingRegressor(random_state=SEED, **gbr_l0a_p).fit(
    train[a_feats], train[target]).predict(test[a_feats])
b_test = GradientBoostingRegressor(random_state=SEED, **gbr_l0b_p).fit(
    train[b_feats], train[target]).predict(test[b_feats])
mgsr_pred = meta.predict(np.column_stack([a_test, b_test]))

fc_rows.append({
    'model': 'MGSR (lagged L0a + static L0b)',
    'R2': r2_score(test[target], mgsr_pred),
    'MAPE': mean_absolute_percentage_error(test[target], mgsr_pred) * 100,
})

# baselines
last_value = pool.sort_values('date').groupby('sub_icb')[target].last()
area_means = pool.groupby('sub_icb')[target].mean()

for name, pred in [
    ('Persistence (last 2023 value)', test.sub_icb.map(last_value)),
    ('Seasonal naive (same month 2023)', test['target_lag12']),
    ('Sub-ICB training mean', test.sub_icb.map(area_means)),
]:
    fc_rows.append({
        'model': name,
        'R2': r2_score(test[target], pred),
        'MAPE': mean_absolute_percentage_error(test[target], pred) * 100,
    })

fc = pd.DataFrame(fc_rows).sort_values('R2', ascending=False)
print(f'\nForecast results for 2024 (n = {len(test)}):')
print(fc.round(4).to_string(index=False))

# monthly errors of the best model vs persistence
best_spec = specs['static + activity lag-1 + y lag-1']
train_b = pool.dropna(subset=best_spec + [target])
best_model = clone(models['GBR']).fit(train_b[best_spec], train_b[target])

test_out = test.copy()
test_out['pred_model'] = best_model.predict(test[best_spec])
test_out['pred_persist'] = test_out.sub_icb.map(last_value)

monthly = test_out.groupby('month_num').agg(
    actual=(target, 'mean'),
    model=('pred_model', 'mean'),
    persistence=('pred_persist', 'mean')
)
monthly['model_err_pct'] = (monthly.model - monthly.actual) / monthly.actual * 100
monthly['persist_err_pct'] = (monthly.persistence - monthly.actual) / monthly.actual * 100
monthly.index = [month_order[i - 1] for i in monthly.index]
print('\nMonthly 2024 errors (%):')
print(monthly.round(2))


Forecast results for 2024 (n = 900):
                                  model     R2    MAPE
 RF (static + activity lag-1 + y lag-1) 0.8621  7.1686
GBR (static + activity lag-1 + y lag-1) 0.8487  7.3287
          Persistence (last 2023 value) 0.7361  9.3246
                  Sub-ICB training mean 0.6879 11.3527
       Seasonal naive (same month 2023) 0.6785 10.8559
                       RF (static only) 0.5757 12.8507
         MGSR (lagged L0a + static L0b) 0.5674 13.6599
                      GBR (static only) 0.5339 13.8611
           RF (static + activity lag-1) 0.5133 14.3463
          GBR (static + activity lag-1) 0.4631 15.1216

Monthly 2024 errors (%):
     actual   model  persistence  model_err_pct  persist_err_pct
Jan  135.12  131.09       132.27          -2.98            -2.11
Feb  127.68  124.05       132.27          -2.84             3.59
Mar  130.97  139.29       132.27           6.36             1.00
Apr  132.04  123.30       132.27          -6.62             0.17
May  1

## 13. Time-series baselines

Fit SARIMA and exponential smoothing separately per Sub-ICB and compare.

In [32]:
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

sarima_pred = []
ets_pred = []
actual = []

for sub in sorted(test.sub_icb.unique()):
    train_series = panel[(panel.sub_icb == sub) & (panel.year <= 2023)].sort_values('date')[target].dropna()
    test_series = test[test.sub_icb == sub].sort_values('date')

    if len(train_series) < 12 or len(test_series) < 6:
        continue

    actual.extend(test_series[target].values)

    try:
        sarima = sm.tsa.SARIMAX(train_series.values, order=(1,1,1),
                                seasonal_order=(1,0,0,12),
                                enforce_stationarity=False,
                                enforce_invertibility=False).fit(disp=False, maxiter=200)
        sarima_pred.extend(sarima.forecast(len(test_series)))
    except Exception:
        sarima_pred.extend([np.nan] * len(test_series))

    try:
        ets = ExponentialSmoothing(train_series.values, trend='add', seasonal=None,
                                   initialization_method='estimated').fit(optimized=True)
        ets_pred.extend(ets.forecast(len(test_series)))
    except Exception:
        ets_pred.extend([np.nan] * len(test_series))

actual = np.array(actual, dtype=float)
sarima_pred = np.array(sarima_pred, dtype=float)
ets_pred = np.array(ets_pred, dtype=float)

print('Time-series baselines:')
for name, pred in [('SARIMA(1,1,1)(1,0,0,12)', sarima_pred),
                   ('ETS (additive trend)', ets_pred)]:
    ok = ~np.isnan(pred)
    r2 = r2_score(actual[ok], pred[ok])
    mape = mean_absolute_percentage_error(actual[ok], pred[ok]) * 100
    print(f'  {name}: R2 = {r2:.4f}, MAPE = {mape:.1f}%')

Time-series baselines:
  SARIMA(1,1,1)(1,0,0,12): R2 = 0.4128, MAPE = 11.1%
  ETS (additive trend): R2 = 0.6226, MAPE = 11.2%


## 14. Figures

### Figure 1 

In [ ]:
# Figure 1: validation schemes (Random 5-fold vs GroupKFold by Sub-ICB vs by ICB)
fig, ax = plt.subplots(figsize=(6.5, 4))
labels = ['Random\n5-fold CV', 'GroupKFold\nby Sub-ICB', 'GroupKFold\nby ICB']
vals   = [best_scores.mean(), r2_group_subicb, r2_group_icb]  
bars = ax.bar(labels, vals, color= '#4C72B0')
ax.axhline(group_mean_scores.mean(), color='#C44E52', linestyle='--',
           label=f'group-mean baseline ({group_mean_scores.mean():.2f})')
ax.axhline(0, color='k', lw=0.8)
for b, v in zip(bars, vals):
    va, offset = ('bottom', 0.02) if v >= 0 else ('top', -0.02)
    ax.text(b.get_x() + b.get_width() / 2, v + offset, f'{v:.2f}',
            ha='center', va=va, fontsize=9)

ax.set_ylabel('$R^2$')
ax.set_ylim(min(vals) - 0.12, 1.0)
ax.legend(loc='upper right')
ax.set_title('Model performance by validation setting')
fig.tight_layout()
fig.savefig(FIG / 'fig2_validation.png', dpi=150)
plt.close(fig)

In [33]:
FIG.mkdir(parents=True, exist_ok=True)

# Figure 1: validation schemes
fig, ax = plt.subplots(figsize=(6.5, 4))
labels = ['Random\n5-fold CV', 'Temporal\nholdout 2024', 'Persistence\nbaseline']
persistence_r2 = fc[fc.model.str.contains('Persistence')].iloc[0].R2
vals = [best_scores.mean(), r2_temporal, persistence_r2]
bars = ax.bar(labels, vals, color=['#4C72B0', '#4C72B0', '#937860'])
ax.axhline(group_mean_scores.mean(), color='#C44E52', linestyle='--',
           label=f'group-mean baseline ({group_mean_scores.mean():.2f})')
ax.axhline(0, color='k', lw=0.8)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}',
            ha='center', fontsize=9)
ax.set_ylabel('$R^2$')
ax.legend()
ax.set_title('Model performance by validation setting')
fig.tight_layout()
fig.savefig(FIG / 'fig1_validation.png', dpi=150)
plt.close(fig)


### Figure 2 

In [34]:
# Figure 2: MGSR stacking
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(l0a_defs))
w = 0.26
l0a_vals, l0b_vals, stack_vals = [], [], []
for cfg, a_feats in l0a_defs.items():
    sa, sb, sm, _ = run_mgsr(a_feats, l0b_feats, gbr_l0a_p, gbr_l0b_p,
                              GradientBoostingRegressor)
    l0a_vals.append(sa.mean())
    l0b_vals.append(sb.mean())
    stack_vals.append(sm.mean())
ax.bar(x - w, l0a_vals, w, label='Level 0a alone')
ax.bar(x, l0b_vals, w, label='Level 0b alone')
ax.bar(x + w, stack_vals, w, label='MGSR stack')
ax.set_xticks(x)
ax.set_xticklabels(list(l0a_defs), fontsize=8)
ax.set_ylabel('$R^2$')
ax.legend()
ax.set_title('Does stacking help? Depends on the Level-0a block')
fig.tight_layout()
fig.savefig(FIG / 'fig2_mgsr.png', dpi=150)
plt.close(fig)


### Figure 3

In [35]:
# Figure 3: predicted vs actual, temporal holdout
pred_t = clone(models['GBR']).fit(train_t[best_feats], train_t[target]).predict(test_t[best_feats])
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test_t[target], pred_t, s=6, alpha=0.35, edgecolor='none')
lo = min(test_t[target].min(), pred_t.min())
hi = max(test_t[target].max(), pred_t.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title(f'Temporal holdout 2024 ($R^2$ = {r2_temporal:.2f})')
fig.tight_layout()
fig.savefig(FIG / 'fig3_pred_vs_actual.png', dpi=150)
plt.close(fig)

### Figure 4 

In [36]:
# Figure 4: monthly 2024
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(12), monthly.actual, 'o-', lw=2, label='actual')
ax.plot(range(12), monthly.model, 's--', label='model')
ax.plot(range(12), monthly.persistence, '^:', label='persistence')
ax.set_xticks(range(12))
ax.set_xticklabels(monthly.index, fontsize=8)
ax.set_ylabel('Mean admissions per 10,000')
ax.legend()
ax.set_title('2024 monthly forecast')
fig.tight_layout()
fig.savefig(FIG / 'fig4_monthly.png', dpi=150)
plt.close(fig)

### Figure 5

In [37]:
# Figure 5: residuals by Sub-ICB
resid = test_t.copy()
resid['r'] = test_t[target].values - pred_t
area_resid = resid.groupby('sub_icb')['r'].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(area_resid)), area_resid.values, color='#4C72B0')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Sub-ICB (sorted)')
ax.set_ylabel('Mean residual')
ax.set_title('Systematic error by Sub-ICB, 2024')
fig.tight_layout()
fig.savefig(FIG / 'fig5_residuals.png', dpi=150)
plt.close(fig)

print('Figures saved to', FIG)

Figures saved to ../figures
